# Berlin Airbnb Data Preparation

This notebook handles the end-to-end data processing pipeline for the Berlin neighborhood and Airbnb listing datasets. It cleanses the raw geometries, processes attributes, performs a spatial join, and exports a standardized GeoPackage for downstream spatial autocorrelation analysis (e.g., Moran's I, Geary's C, Getis-Ord G, and Join Counts).

## 1. Environment Setup

In [14]:
import geopandas as gpd
import pandas as pd

## 2. Load Raw Data

We read the neighborhood boundary vector layers alongside the tabular point-source Airbnb listings.

In [15]:
# Load neighborhood polygons
gdf = gpd.read_file("berlin-neighbourhoods.geojson")

# Load continuous Airbnb listings metadata
bl_df = pd.read_csv("berlin-listings.csv")
print(f"Loaded {len(gdf)} neighborhoods and {len(bl_df)} individual point listings.")

Loaded 140 neighborhoods and 20053 individual point listings.


## 3. Geometric Transformation

Convert the tabular coordinate pairs into explicit point geometry coordinates assigned to standard WGS84 (`EPSG:4326`).

In [16]:
geometry = gpd.points_from_xy(x=bl_df.longitude, y=bl_df.latitude, crs="epsg:4326")
bl_gdf = gpd.GeoDataFrame(bl_df, geometry=geometry)

# Harmonize attribute data types to prevent structural calculation issues
bl_gdf["price"] = bl_gdf["price"].astype("float32")

## 4. Spatial Join and Feature Engineering

We run a spatial intersection (`intersects`) to map point listings into their matching home neighborhood boundaries, calculate the aggregate mean across neighborhood groups, and clean out missing values or structural ties.

In [ ]:
# Execute inner spatial join
sj_gdf = gpd.sjoin(
    gdf, bl_gdf, how="inner", predicate="intersects", lsuffix="left", rsuffix="right"
)

# Calculate the mean price grouped by neighborhood boundaries
median_price_gb = sj_gdf["price"].groupby([sj_gdf["neighbourhood_group"]]).mean()

# Merge statistical aggregations back to the master polygon collection
gdf = gdf.join(median_price_gb, on="neighbourhood_group")
gdf.rename(columns={"price": "median_pri"}, inplace=True)

# Handle missing data (NaNs) using global study-wide mean imputation
global_mean = gdf["median_pri"].mean()
gdf["median_pri"] = gdf["median_pri"].fillna(global_mean)

print(f"Imputed missing fields using global mean value: {global_mean:.4f}")

## 5. Categorical Dichotomization

To support nominal analysis (like Join Counts), we construct an explicit binary indicator column flag (`yb_binary`) mapping regions sitting above (`1`) or below (`0`) the global median value.

In [18]:
median_val = gdf["median_pri"].median()
gdf["yb_binary"] = (gdf["median_pri"] > median_val).astype(int)
msg = "Dichotomized around median price"
high = sum(gdf["yb_binary"])
low = len(gdf) - sum(gdf["yb_binary"])
print(f"Dichotomized  ({median_val:.2f}): {high} High, {low} Low.")

Dichotomized  (53.70): 68 High, 72 Low.


## 6. Export Clean Dataset

Save out the standardized operational dataset directly to an OGC GeoPackage format (`.gpkg`). This acts as our single source of truth for all analytical spatial models.

In [19]:
output_path = "berlin-housing.gpkg"

# Drop transient or intermediate index join columns if present to stay clean
gdf = gdf.loc[:, ~gdf.columns.duplicated()]
if "index_right" in gdf.columns:
    gdf = gdf.drop(columns=["index_right"])

# Export geometry dataframe
gdf.to_file(output_path, layer="neighbourhoods", driver="GPKG")
print(f"Successfully written clean dataset to: {output_path}")

Successfully written clean dataset to: berlin-housing.gpkg
